<!-- # core

> Fill in a module description here -->

# core
> The core module provides the foundation for working with linked data in JSON-LD format. It serves as the memory system for LLM agents, enabling them to store, organize, and retrieve knowledge effectively.



In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

## Key Features

- **Flexible Knowledge Organization**: Store knowledge in multiple structures (main graph, named graphs, included entities)
- **JSON-LD 1.1 Support**: Use advanced features like `@container` and `@included` for semantic organization
- **Entity Management**: Find, relate, and navigate between entities across different knowledge structures
- **Memory Management**: Create separate "memory spaces" for different domains or sources
- **Agent-Friendly Design**: Intuitive structure that LLMs can navigate and reason about

## Memory Architecture

The LinkedDataKnowledge class implements a multi-level memory architecture:

1. **Main Graph**: The default flat structure for entities in `@graph`
2. **Named Graphs**: Separate knowledge graphs with metadata, stored in `self.graphs`
3. **Included Entities**: Resource-centric organization using JSON-LD 1.1's `@included`
4. **Container Collections**: Specialized collections using JSON-LD 1.1 container types

This architecture supports both human-readable organization and machine-processable semantics, making it ideal for LLM agents that need to work with structured knowledge.

## Usage Patterns

- **Knowledge Organization**: Use named graphs for separate domains, `@included` for entity-centric views
- **Knowledge Navigation**: Find entities across graphs, follow relationships, explore neighborhoods
- **Memory Management**: Add, retrieve, and organize knowledge in semantically meaningful ways
- **LLM Integration**: Provide rich, structured context for LLM reasoning and decision-making

The core module is designed to be extended by the vocabulary, navigation, and dataset modules, which build on these foundational capabilities for specific use cases.

In [ ]:
#| export

from fastcore.basics import *
from fastcore.meta import *
from fastcore.test import *
import json
from rdflib import Graph
from pyld import jsonld
from typing import List, Dict, Any, Optional, Union
import datetime

In [ ]:
#| hide
from IPython.display import display, Markdown, HTML

In [ ]:
#| export
# Add this after the imports
MEMORY_CONTEXT = {
    "@context": {
        "schema": "https://schema.org/",
        "dcat": "http://www.w3.org/ns/dcat#",
        "dcterms": "http://purl.org/dc/terms/",
        "prov": "http://www.w3.org/ns/prov#",
        "void": "http://rdfs.org/ns/void#",
        "did": "https://www.w3.org/ns/did/v1#",
        
        "KnowledgeBase": "schema:Dataset",
        "graphs": {
            "@id": "void:subset",
            "@container": "@id"
        },
        "source": {
            "@id": "dcterms:source",
            "@type": "@id"
        },
        "controller": {
            "@id": "did:controller",
            "@type": "@id"
        },
        "description": "dcterms:description",
        "entityCount": "void:entities",
        "lastUpdated": {
            "@id": "dcterms:modified",
            "@type": "xsd:dateTime"
        },
        "title": "dcterms:title",
        "vocabulary": {
            "@id": "void:vocabulary",
            "@type": "@id"
        }
    }
}

## Memory Management for LLM Agents

The LinkedDataKnowledge class serves as a memory management system for LLM agents, providing several ways to organize and access knowledge:

1. **Main Graph**: The default `@graph` array stores entities in a flat structure
2. **Named Graphs**: The `graphs` dictionary stores separate knowledge graphs with metadata
3. **Included Entities**: The `@included` array supports JSON-LD 1.1's resource-centric view

This flexible structure allows LLMs to:
- Maintain separate "memory spaces" for different domains
- Follow relationships between entities within and across graphs
- Organize knowledge by source, topic, or importance
- Retrieve specific entities or explore entire knowledge areas

The memory system is designed to be intuitive for LLMs to navigate while maintaining compatibility with standard JSON-LD tools and processors.

In [ ]:
#| export
class LinkedDataKnowledge:
    "Represents a knowledge base of linked data in JSON-LD format"
    def __init__(self, 
                 data:Dict=None, # Initial knowledge data
                ):
        self.data = data or {"@context": {}, "@graph": []}
        
    def __repr__(self): return f"LinkedDataKnowledge with {len(self.data.get('@graph', []))} entities"

In [ ]:
# Create an empty knowledge base
kb = LinkedDataKnowledge()
kb

# Create with sample data
sample_data = {
    "@context": {"@vocab": "http://schema.org/"},
    "@graph": [
        {"@id": "http://example.org/person1", "@type": "Person", "name": "Alice"},
        {"@id": "http://example.org/person2", "@type": "Person", "name": "Bob"}
    ]
}
kb_with_data = LinkedDataKnowledge(sample_data)
kb_with_data

LinkedDataKnowledge with 2 entities

In [ ]:
#| export
@patch
def _repr_markdown_(self:LinkedDataKnowledge) -> str:
    "Rich markdown representation for notebook display"
    md = [f"## LinkedDataKnowledge"]
    
    # Context summary
    context = self.data.get('@context', {})
    md.append(f"### Context ({len(context)} prefixes)")
    
    # Show the first few context entries
    if context:
        md.append("```json")
        context_preview = dict(list(context.items())[:5])
        if len(context) > 5:
            md.append(json.dumps(context_preview, indent=2) + "\n... and more")
        else:
            md.append(json.dumps(context, indent=2))
        md.append("```")
    
    # Graph summary
    graph = self.data.get('@graph', [])
    md.append(f"### Graph ({len(graph)} entities)")
    
    # Show types of entities in the graph
    if graph:
        types = {}
        for entity in graph:
            entity_type = entity.get('@type')
            if isinstance(entity_type, list):
                for t in entity_type:
                    types[t] = types.get(t, 0) + 1
            elif entity_type:
                types[entity_type] = types.get(entity_type, 0) + 1
        
        if types:
            md.append("**Entity types:**")
            for t, count in sorted(types.items(), key=lambda x: x[1], reverse=True):
                md.append(f"- {t}: {count}")
    
    # Show a sample entity if available
    if graph:
        md.append("\n**Sample entity:**")
        md.append("```json")
        md.append(json.dumps(graph[0], indent=2))
        md.append("```")
    
    # If using @included, show that too
    if '@included' in self.data:
        included = self.data['@included']
        md.append(f"\n### Included ({len(included)} entities)")
        md.append("**Types of included entities:**")
        
        # Count types of included entities
        included_types = {}
        for entity in included:
            entity_type = entity.get('@type')
            if isinstance(entity_type, list):
                for t in entity_type:
                    included_types[t] = included_types.get(t, 0) + 1
            elif entity_type:
                included_types[entity_type] = included_types.get(entity_type, 0) + 1
        
        for t, count in sorted(included_types.items(), key=lambda x: x[1], reverse=True):
            md.append(f"- {t}: {count}")
    
    return "\n".join(md)


In [ ]:
#| export
def _format_entity_markdown(entity:Dict) -> str:
    "Format a single entity as markdown"
    md = []
    
    # Entity ID and type
    entity_id = entity.get('@id', 'No ID')
    entity_type = entity.get('@type', ['Unknown'])
    if isinstance(entity_type, list):
        entity_type = ', '.join(entity_type)
    
    md.append(f"### {entity_type}: {entity_id}")
    
    # Properties
    for prop, values in entity.items():
        if prop in ['@id', '@type']:
            continue
            
        # Format the property name
        prop_name = prop.split('/')[-1] if '/' in prop else prop
        md.append(f"**{prop_name}**:")
        
        # Format values
        for val in values:
            if isinstance(val, dict):
                if '@value' in val:
                    value_text = val['@value']
                    if '@language' in val:
                        value_text += f" @{val['@language']}"
                    md.append(f"- {value_text}")
                elif '@id' in val:
                    md.append(f"- [{val['@id']}]({val['@id']})")
                else:
                    md.append(f"- {val}")
            else:
                md.append(f"- {val}")
    
    return "\n".join(md)

@patch
def query_markdown(self:LinkedDataKnowledge,
                  query_type:str, # Type of query: "property", "type", "value" 
                  query_value:str, # Value to search for
                  ) -> str:
    "Query the knowledge base and return results as markdown"
    results = self.query(query_type, query_value)
    
    if not results:
        return f"*No results found for {query_type}='{query_value}'*"
    
    md = [f"# Query Results: {query_type}='{query_value}'", 
          f"Found {len(results)} matching entities"]
    
    # Format each result entity
    for i, entity in enumerate(results[:5]):  # Limit to first 5 for readability
        md.append(f"\n## Result {i+1}")
        md.append(_format_entity_markdown(entity))
    
    if len(results) > 5:
        md.append(f"\n*...and {len(results)-5} more results*")
    
    return "\n".join(md)


In [ ]:
#| export
@patch
def initialize_memory_structure(self:LinkedDataKnowledge) -> 'LinkedDataKnowledge':
    "Initialize the knowledge base with a standard memory structure"
    if not hasattr(self, 'graphs'):
        self.graphs = {}
    
    self.data = {
        "@context": MEMORY_CONTEXT["@context"],
        "@id": "did:cogitarelink:kb",
        "@type": "KnowledgeBase",
        "title": "Cogitarelink Knowledge Base",
        "description": "A decentralized knowledge system for AI agent reasoning",
        "controller": "did:cogitarelink:agent:primary",
        "graphs": {}
    }
    
    return self

In [ ]:
# Test initialize_memory_structure
kb = LinkedDataKnowledge()
kb.initialize_memory_structure()

# Test that graphs container was created
test_eq(hasattr(kb, 'graphs'), True)
test_eq(isinstance(kb.graphs, dict), True)

# Test data structure initialization
test_eq(kb.data["@id"], "did:cogitarelink:kb")
test_eq(kb.data["@type"], "KnowledgeBase")
test_eq("@context" in kb.data, True)
test_eq("graphs" in kb.data, True)


In [ ]:
#| export
@patch
def add_named_graph(self:LinkedDataKnowledge, 
                   graph_id:str, # DID or URI for the graph
                   graph_data:dict, # JSON-LD data for the graph
                   metadata:dict=None # Optional metadata about the graph
                  ) -> 'LinkedDataKnowledge':
    "Add a named graph to the knowledge base"
    if not hasattr(self, 'graphs'):
        self.initialize_memory_structure()
    
    if not graph_id.startswith(('did:', 'http://', 'https://')):
        graph_id = f"did:cogitarelink:graph:{graph_id}"
    
    if metadata is None:
        metadata = {}
    
    entity_count = len(graph_data.get('@graph', []))
    
    if 'title' not in metadata:
        metadata['title'] = f"Graph {graph_id.split(':')[-1]}"
    if 'description' not in metadata:
        metadata['description'] = f"Graph containing {entity_count} entities"
    if 'entityCount' not in metadata:
        metadata['entityCount'] = entity_count
    if 'lastUpdated' not in metadata:
        metadata['lastUpdated'] = datetime.datetime.now().isoformat()
    
    self.graphs[graph_id] = {
        'data': graph_data,
        'metadata': metadata
    }
    
    self.data['graphs'][graph_id] = {
        "@id": graph_id,
        "@type": "void:Dataset",
        **metadata
    }
    
    return self

#| export
@patch
def get_named_graph(self:LinkedDataKnowledge, 
                   graph_id:str # DID or URI for the graph
                  ) -> dict:
    "Retrieve a named graph from the knowledge base"
    if not hasattr(self, 'graphs'):
        return None
    
    if not graph_id.startswith(('did:', 'http://', 'https://')):
        graph_id = f"did:cogitarelink:graph:{graph_id}"
    
    if graph_id in self.graphs:
        return self.graphs[graph_id]['data']
    
    return None


# Working with Named Graphs

Named graphs allow an LLM agent to organize its knowledge into separate but connected structures. Each graph has:

1. A unique identifier (typically a DID like `did:cogitarelink:graph:topic`)
2. JSON-LD data containing entities related to a specific topic or source
3. Metadata describing the graph's contents and provenance

### When to Use Named Graphs

- **Different knowledge domains**: Separate scientific knowledge from cultural knowledge
- **Different data sources**: Keep Wikidata entities separate from Schema.org vocabulary
- **Different confidence levels**: Distinguish between verified facts and uncertain information
- **Memory organization**: Create "memory palaces" for different topics

### Example Usage

```python
# Create a knowledge base with multiple graphs
kb = LinkedDataKnowledge()
kb.initialize_memory_structure()

# Add a graph about physics
kb.add_named_graph(
    "physics",
    {"@graph": [{"@id": "ex:relativity", "@type": "Theory"}]},
    {"title": "Physics Concepts", "source": "textbook"}
)

# Add a graph from Wikidata
kb.add_named_graph(
    "einstein",
    wikidata_data,
    {"title": "Einstein Biography", "source": "Wikidata"}
)

# Retrieve a specific graph
physics_graph = kb.get_named_graph("physics")

In [ ]:
# Create a simple test graph
test_graph = {
    "@context": {"ex": "http://example.org/"},
    "@graph": [
        {"@id": "ex:entity1", "@type": "ex:TestEntity", "ex:name": "Test Entity 1"},
        {"@id": "ex:entity2", "@type": "ex:TestEntity", "ex:name": "Test Entity 2"}
    ]
}

# Test adding a graph with explicit metadata
kb.add_named_graph("test1", test_graph, {"title": "Test Graph 1"})

# Check the graph was properly added
graph_id = "did:cogitarelink:graph:test1"
test_eq(graph_id in kb.graphs, True)
test_eq(kb.graphs[graph_id]["data"], test_graph)
test_eq(kb.graphs[graph_id]["metadata"]["title"], "Test Graph 1")

# Check graph reference in main data structure
test_eq(graph_id in kb.data["graphs"], True)
test_eq(kb.data["graphs"][graph_id]["title"], "Test Graph 1")

# Test adding a graph with default metadata
kb.add_named_graph("test2", test_graph)

# Check default metadata generation
graph_id2 = "did:cogitarelink:graph:test2"
test_eq(kb.graphs[graph_id2]["metadata"]["title"], "Graph test2")
test_eq(kb.graphs[graph_id2]["metadata"]["entityCount"], 2)

# Test graph retrieval
retrieved_graph = kb.get_named_graph("test1")
test_eq(retrieved_graph, test_graph)

# Test retrieving non-existent graph
nonexistent_graph = kb.get_named_graph("nonexistent")
test_eq(nonexistent_graph, None)


In [ ]:
#| export
@patch
def use_included(self:LinkedDataKnowledge,
                main_entity:Dict, # Main entity to focus on
                related_entities:List[Dict], # Related entities to include
                preserve_structure:bool=False # Whether to preserve existing structure
                ) -> 'LinkedDataKnowledge':
    "Structure knowledge using @included for related entities"
    if preserve_structure:
        new_data = {k: v for k, v in self.data.items() if k not in ['@included']}
        new_data['@included'] = related_entities
        
        if '@graph' in new_data:
            main_id = main_entity.get('@id')
            if main_id:
                existing_entities = [e for e in new_data['@graph'] if e.get('@id') == main_id]
                if existing_entities:
                    for entity in new_data['@graph']:
                        if entity.get('@id') == main_id:
                            for k, v in main_entity.items():
                                entity[k] = v
                else:
                    new_data['@graph'].append(main_entity)
            else:
                new_data['@graph'].append(main_entity)
        else:
            new_data['@graph'] = [main_entity]
    else:
        new_data = {
            "@context": self.data.get("@context", {}),
            **main_entity,
            "@included": related_entities
        }
    
    self.data = new_data
    return self

In [ ]:
# Test use_included
kb_included = LinkedDataKnowledge()

# Create main entity and related entities
main_entity = {
    "@id": "ex:main",
    "@type": "ex:MainEntity",
    "ex:name": "Main Entity"
}

related_entities = [
    {
        "@id": "ex:related1",
        "@type": "ex:RelatedEntity",
        "ex:name": "Related Entity 1"
    },
    {
        "@id": "ex:related2",
        "@type": "ex:RelatedEntity",
        "ex:name": "Related Entity 2"
    }
]

# Test with replace structure (default)
kb_included.use_included(main_entity, related_entities)

# Check that main entity properties are at root level
test_eq(kb_included.data["@id"], "ex:main")
test_eq(kb_included.data["@type"], "ex:MainEntity")

# Check that related entities are in @included
test_eq("@included" in kb_included.data, True)
test_eq(len(kb_included.data["@included"]), 2)
test_eq(kb_included.data["@included"][0]["@id"], "ex:related1")

# Test with preserve structure
kb_preserve = LinkedDataKnowledge()
kb_preserve.data = {
    "@context": {},
    "@graph": [
        {"@id": "ex:existing", "@type": "ex:ExistingEntity"}
    ],
    "customProperty": "Custom Value"
}

kb_preserve.use_included(main_entity, related_entities, preserve_structure=True)

# Check that existing structure is preserved
test_eq("customProperty" in kb_preserve.data, True)
test_eq(kb_preserve.data["customProperty"], "Custom Value")

# Check that main entity is added to @graph
test_eq(len(kb_preserve.data["@graph"]), 2)
test_eq(any(e["@id"] == "ex:main" for e in kb_preserve.data["@graph"]), True)

# Check that related entities are in @included
test_eq("@included" in kb_preserve.data, True)
test_eq(len(kb_preserve.data["@included"]), 2)


In [ ]:
#| export
@patch
def structure_with_containers(self:LinkedDataKnowledge,
                             property_name:str, # Property to structure
                             container_type:str, # Type: "set", "list", "language", "id", "type"
                             items:List=None, # Items to include, or None to use existing
                             base_uri:str="https://example.org/" # Base URI for properties
                             ) -> 'LinkedDataKnowledge':
    "Structure a property using JSON-LD 1.1 container features"
    container_mapping = {
        "set": "@set",
        "list": "@list",
        "language": "@language",
        "id": "@id",
        "type": "@type"
    }
    
    container = container_mapping.get(container_type)
    if not container:
        raise ValueError(f"Unknown container type: {container_type}")
    
    container_context = {
        "@version": 1.1,
        property_name: {
            "@id": f"{base_uri}{property_name}",
            "@container": container
        }
    }
    
    if '@context' not in self.data:
        self.data['@context'] = {}
    self.data['@context'].update(container_context)
    
    if items is not None:
        if container_type in ["set", "list"]:
            self.data[property_name] = items
        elif container_type == "language":
            self.data[property_name] = {item["lang"]: item["value"] for item in items}
        elif container_type == "id":
            self.data[property_name] = {item["id"]: item["value"] for item in items}
        elif container_type == "type":
            self.data[property_name] = {item["type"]: item["value"] for item in items}
    
    return self


## JSON-LD 1.1 Features for Knowledge Organization

LinkedDataKnowledge supports advanced JSON-LD 1.1 features that help LLM agents organize information more effectively:

### Resource-Centric View with @included

The `use_included` method implements JSON-LD 1.1's resource-centric view, which organizes data around a main entity with related entities in an `@included` array. This is useful for:

- Creating entity-focused knowledge structures
- Maintaining focus on a primary topic while including supporting information
- Organizing knowledge in a more narrative-friendly way

### Containers for Structured Collections

The `structure_with_containers` method uses JSON-LD 1.1 container types to create specialized collections:

- `@set`: Unordered collections of values
- `@list`: Ordered collections with preserved order
- `@language`: Collections indexed by language tag
- `@id`: Collections indexed by ID
- `@type`: Collections indexed by type

These containers help LLMs organize and access information in more semantically meaningful ways.

In [ ]:
# Test structure_with_containers with different container types
kb_containers = LinkedDataKnowledge()

# Test @list container
list_items = ["Item 1", "Item 2", "Item 3"]
kb_containers.structure_with_containers("steps", "list", list_items)

# Check context definition
test_eq("@version" in kb_containers.data["@context"], True)
test_eq("steps" in kb_containers.data["@context"], True)
test_eq(kb_containers.data["@context"]["steps"]["@container"], "@list")

# Check list values
test_eq("steps" in kb_containers.data, True)
test_eq(kb_containers.data["steps"], list_items)

# Test @language container
language_items = [
    {"lang": "en", "value": "Hello"},
    {"lang": "es", "value": "Hola"},
    {"lang": "fr", "value": "Bonjour"}
]
kb_containers.structure_with_containers("greeting", "language", language_items)

# Check context definition
test_eq(kb_containers.data["@context"]["greeting"]["@container"], "@language")

# Check language map values
test_eq("greeting" in kb_containers.data, True)
test_eq(kb_containers.data["greeting"]["en"], "Hello")
test_eq(kb_containers.data["greeting"]["es"], "Hola")
test_eq(kb_containers.data["greeting"]["fr"], "Bonjour")

# Test @id container
id_items = [
    {"id": "person1", "value": {"name": "Alice"}},
    {"id": "person2", "value": {"name": "Bob"}}
]
kb_containers.structure_with_containers("people", "id", id_items)

# Check context definition
test_eq(kb_containers.data["@context"]["people"]["@container"], "@id")

# Check id map values
test_eq("people" in kb_containers.data, True)
test_eq(kb_containers.data["people"]["person1"]["name"], "Alice")
test_eq(kb_containers.data["people"]["person2"]["name"], "Bob")


In [ ]:
#| export
@patch
def find_entity_across_graphs(self:LinkedDataKnowledge, 
                             entity_id:str=None, # Full or partial entity ID to find
                             term_type:str=None, # Filter by type (e.g., "Class", "Property")
                             label:str=None, # Find by label text
                             graph_id:str=None, # Specific graph to search, or None for all
                             include_main_graph:bool=True # Whether to include the main graph
                            ) -> list:
    "Find entities across all graphs or in a specific graph"
    results = []
    
    if include_main_graph:
        main_results = self.find_entity(entity_id, term_type, label)
        results.extend(main_results)
    
    if hasattr(self, 'graphs'):
        graphs_to_search = [graph_id] if graph_id else self.graphs.keys()
        
        for gid in graphs_to_search:
            if gid in self.graphs:
                graph_data = self.graphs[gid]['data']
                temp_kb = LinkedDataKnowledge(graph_data)
                graph_results = temp_kb.find_entity(entity_id, term_type, label)
                results.extend(graph_results)
    
    return results

## Entity Finding Across Knowledge Structures

The `find_entity_across_graphs` method allows LLMs to search for entities across multiple knowledge structures:

- Search in the main graph, named graphs, or both
- Filter by entity ID, type, or label
- Focus on specific graphs for domain-specific searches

This capability is essential for LLMs to navigate their knowledge effectively, especially as their knowledge base grows more complex over time.

### Example Usage

```python
# Find Einstein across all graphs
einstein_entities = kb.find_entity_across_graphs(entity_id="Einstein")

# Find only in the physics graph
physics_concepts = kb.find_entity_across_graphs(
    term_type="Theory", 
    graph_id="did:cogitarelink:graph:physics",
    include_main_graph=False
)

# Find by label across all structures
relativity_entities = kb.find_entity_across_graphs(label="relativity")

In [ ]:
# Test find_entity_across_graphs
kb_search = LinkedDataKnowledge()
kb_search.initialize_memory_structure()

# Add entities to main graph
kb_search.data["@graph"] = [
    {"@id": "ex:main1", "@type": "ex:MainEntity", "rdfs:label": "Main Entity 1"},
    {"@id": "ex:main2", "@type": "ex:MainEntity", "rdfs:label": "Main Entity 2"}
]

# Add a named graph with more entities
graph1_data = {
    "@graph": [
        {"@id": "ex:graph1_entity1", "@type": "ex:GraphEntity", "rdfs:label": "Graph 1 Entity 1"},
        {"@id": "ex:graph1_entity2", "@type": "ex:GraphEntity", "rdfs:label": "Graph 1 Entity 2"}
    ]
}
kb_search.add_named_graph("graph1", graph1_data)

# Add another named graph
graph2_data = {
    "@graph": [
        {"@id": "ex:graph2_entity1", "@type": "ex:GraphEntity", "rdfs:label": "Graph 2 Entity 1"},
        {"@id": "ex:main1", "@type": "ex:DuplicateEntity", "rdfs:label": "Duplicate Main Entity"}
    ]
}
kb_search.add_named_graph("graph2", graph2_data)

# Test search across all graphs
all_entities = kb_search.find_entity_across_graphs(term_type="ex:GraphEntity")
test_eq(len(all_entities), 3)  # Should find all 3 GraphEntity instances

# Test search in specific graph
graph1_entities = kb_search.find_entity_across_graphs(
    term_type="ex:GraphEntity",
    graph_id="did:cogitarelink:graph:graph1",
    include_main_graph=False
)
test_eq(len(graph1_entities), 2)  # Should find only the 2 in graph1

# Test finding by ID across graphs
main_entities = kb_search.find_entity_across_graphs(entity_id="ex:main1")
test_eq(len(main_entities), 2)  # Should find in both main graph and graph2

# Test finding by label
label_entities = kb_search.find_entity_across_graphs(label="Main Entity")
test_eq(len(label_entities), 3)  # Updated: finds 3 entities with "Main Entity" in the label


AttributeError: 'LinkedDataKnowledge' object has no attribute 'find_entity'

## This test exposes a potential issue with partial label matching

- "Main Entity 1" from the main graph
- "Main Entity 2" from the main graph
- "Duplicate Main Entity" from graph2
The test was expecting only 2, but the partial match for "Main Entity" is finding all 3 entities. I've updated the test to expect 3 results instead of 2.

Alternatively, if you want to test for exact label matches only, we could modify the test data or the search criteria:

### Alternative test for exact label matches
```python
exact_label_entities = kb_search.find_entity_across_graphs(label="Main Entity 1")
test_eq(len(exact_label_entities), 1)  # Should find exactly one entity with this exact lab

In [ ]:
#| export
@patch
def _has_type(self:LinkedDataKnowledge, entity:dict, type_str:str) -> bool:
    """Check if entity has the specified type, handling various type formats."""
    if not entity or not type_str:
        return False
        
    entity_types = entity.get('@type', [])
    if not isinstance(entity_types, list):
        entity_types = [entity_types]
    
    # Handle empty types
    if not entity_types:
        return False
    
    # Try exact match first (most efficient)
    if type_str in entity_types:
        return True
    
    # For prefixed names like "rdfs:Class" or "rdf:Property", try direct string matching
    # This handles cases where the prefix isn't in the context but is used in entity types
    if ':' in type_str and any(t.endswith(type_str.split(':')[1]) for t in entity_types):
        return True
    
    # Prepare for URI and prefixed name handling
    context = self.data.get('@context', {})
    
    # Standard RDF prefixes that might not be in the context
    standard_prefixes = {
        "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
        "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
        "owl": "http://www.w3.org/2002/07/owl#",
        "xsd": "http://www.w3.org/2001/XMLSchema#"
    }
    
    # Add standard prefixes to context if not already present
    for prefix, uri in standard_prefixes.items():
        if prefix not in context:
            context[prefix] = uri
    
    # Convert type_str to expanded URI if it's a prefixed name
    expanded_type_str = None
    if ':' in type_str and not type_str.startswith(('http://', 'https://')):
        prefix, local = type_str.split(':', 1)
        if prefix in context:
            prefix_uri = context[prefix]
            if isinstance(prefix_uri, str):
                if prefix_uri.endswith('/') or prefix_uri.endswith('#'):
                    expanded_type_str = f"{prefix_uri}{local}"
                else:
                    expanded_type_str = f"{prefix_uri}/{local}"
    
    # Convert entity_types to expanded URIs and prefixed names for comparison
    for entity_type in entity_types:
        # Check if expanded type_str matches entity_type
        if expanded_type_str and (expanded_type_str == entity_type or expanded_type_str in entity_type):
            return True
            
        # If entity_type is a full URI and type_str is a prefixed name
        if entity_type.startswith(('http://', 'https://')) and ':' in type_str and not type_str.startswith(('http://', 'https://')):
            # Extract the local name from the URI to match against prefixed name
            for prefix, uri in context.items():
                if isinstance(uri, str) and entity_type.startswith(uri):
                    local_part = entity_type[len(uri):]
                    if local_part.startswith('/') or local_part.startswith('#'):
                        local_part = local_part[1:]
                    prefixed = f"{prefix}:{local_part}"
                    if prefixed == type_str:
                        return True
                        
        # If type_str is a full URI and entity_type is a prefixed name
        if type_str.startswith(('http://', 'https://')) and ':' in entity_type and not entity_type.startswith(('http://', 'https://')):
            prefix, local = entity_type.split(':', 1)
            if prefix in context:
                prefix_uri = context[prefix]
                if isinstance(prefix_uri, str):
                    if prefix_uri.endswith('/') or prefix_uri.endswith('#'):
                        expanded = f"{prefix_uri}{local}"
                    else:
                        expanded = f"{prefix_uri}/{local}"
                    if expanded == type_str or expanded in type_str:
                        return True
    
    # Check for local name match (e.g., "Class" matches "rdfs:Class" or "http://.../Class")
    if not type_str.startswith(('http://', 'https://')) and not ':' in type_str:
        # Check if any type ends with the local name
        if any(t.split('/')[-1] == type_str or 
               t.split('#')[-1] == type_str or
               ((':' in t) and t.split(':')[-1] == type_str)
               for t in entity_types):
            return True
    
    # Finally, try substring match as a fallback
    # This is less reliable but catches some edge cases
    return any(type_str in t for t in entity_types)


In [ ]:
#| export
@patch
def display_entity(self:LinkedDataKnowledge, entity_id:str) -> str:
    "Display a specific entity by ID as markdown"
    
    # First try in @graph
    for entity in self.data.get('@graph', []):
        if entity.get('@id') == entity_id:
            return _format_entity_markdown(entity)
    
    # Then try in @included if present
    for entity in self.data.get('@included', []):
        if entity.get('@id') == entity_id:
            return _format_entity_markdown(entity)
    
    # If it's the main entity in a resource-centric view
    if self.data.get('@id') == entity_id:
        return _format_entity_markdown(self.data)
    
    return f"*Entity with ID '{entity_id}' not found*"

@patch
def summarize_markdown(self:LinkedDataKnowledge) -> str:
    "Provide a concise markdown summary of the knowledge base"
    
    md = ["# Knowledge Base Summary"]
    
    # Count contexts, entities, and included entities
    context_count = len(self.data.get('@context', {}))
    graph_count = len(self.data.get('@graph', []))
    included_count = len(self.data.get('@included', [])) if '@included' in self.data else 0
    
    md.append(f"- **Contexts:** {context_count}")
    md.append(f"- **Graph Entities:** {graph_count}")
    if included_count:
        md.append(f"- **Included Entities:** {included_count}")
    
    # Summarize entity types
    all_types = {}
    
    # Check @graph
    for entity in self.data.get('@graph', []):
        entity_type = entity.get('@type')
        if isinstance(entity_type, list):
            for t in entity_type:
                all_types[t] = all_types.get(t, 0) + 1
        elif entity_type:
            all_types[entity_type] = all_types.get(entity_type, 0) + 1
    
    # Check @included
    for entity in self.data.get('@included', []):
        entity_type = entity.get('@type')
        if isinstance(entity_type, list):
            for t in entity_type:
                all_types[t] = all_types.get(t, 0) + 1
        elif entity_type:
            all_types[entity_type] = all_types.get(entity_type, 0) + 1
    
    # Check main entity if resource-centric
    if '@id' in self.data and '@type' in self.data:
        entity_type = self.data.get('@type')
        if isinstance(entity_type, list):
            for t in entity_type:
                all_types[t] = all_types.get(t, 0) + 1
        elif entity_type:
            all_types[entity_type] = all_types.get(entity_type, 0) + 1
    
    if all_types:
        md.append("\n## Entity Types")
        for t, count in sorted(all_types.items(), key=lambda x: x[1], reverse=True)[:10]:
            md.append(f"- **{t}**: {count}")
        if len(all_types) > 10:
            md.append(f"- *...and {len(all_types)-10} more types*")
    
    return "\n".join(md)


In [ ]:
#| export
@patch
def find_entity(self:LinkedDataKnowledge, 
               entity_id:str=None, # Full or partial entity ID to find
               term_type:str=None, # Filter by type (e.g., "Class", "Property")
               label:str=None, # Find by label text
               prioritize_label:bool=True, # Whether to prioritize label matches over ID matches
               case_sensitive:bool=False # Whether searches should be case sensitive
              ) -> list: # Matching entities
    """Find entities in the graph by ID, type, or label.
    
    This function provides flexible entity lookup by:
    - ID (full URI, prefixed name, or local name)
    - Type (class or property type)
    - Label (exact or partial match)
    
    Args:
        entity_id: Full or partial entity ID to find
        term_type: Filter by type (e.g., "Class", "Property")
        label: Find by label text
        prioritize_label: Whether to prioritize label matches over ID matches
        case_sensitive: Whether searches should be case sensitive
        
    Returns:
        list: Matching entities
    """
    results = []
    label_results = []
    id_results = []
    graph = self.data.get('@graph', [])
    
    # Handle case sensitivity
    if entity_id and not case_sensitive:
        entity_id = entity_id.lower()
    if label and not case_sensitive:
        label = label.lower()
    
    # Process each entity, checking ID, type, and label as needed
    for entity in graph:
        # First check if entity matches the type filter (if specified)
        if term_type and not self._has_type(entity, term_type):
            continue  # Skip entities that don't match the type filter
        
        # Track if this entity matched any criteria
        matched = False
        
        # Match by ID if specified
        if entity_id and isinstance(entity.get('@id'), str):
            entity_uri = entity.get('@id')
            
            # Handle case sensitivity for URI
            if not case_sensitive:
                compare_uri = entity_uri.lower()
            else:
                compare_uri = entity_uri
            
            # Check for exact match, substring match, or path segment match
            if (entity_id == compare_uri or
                entity_id in compare_uri or
                compare_uri.split('/')[-1] == entity_id or
                compare_uri.split('#')[-1] == entity_id):
                
                id_results.append(entity)
                matched = True
        
        # Match by label (even if we already matched by ID)
        if label or (prioritize_label and entity_id):
            search_term = label if label else entity_id
            
            for key, value in entity.items():
                if 'label' in key.lower():
                    # Handle different label formats
                    if isinstance(value, list):
                        for item in value:
                            if isinstance(item, dict) and '@value' in item:
                                item_value = str(item.get('@value', ''))
                                if not case_sensitive:
                                    item_value = item_value.lower()
                                
                                if search_term in item_value:
                                    label_results.append(entity)
                                    matched = True
                                    break
                            else:
                                item_str = str(item)
                                if not case_sensitive:
                                    item_str = item_str.lower()
                                    
                                if search_term in item_str:
                                    label_results.append(entity)
                                    matched = True
                                    break
                    elif isinstance(value, str):
                        if not case_sensitive:
                            compare_value = value.lower()
                        else:
                            compare_value = value
                            
                        if search_term in compare_value:
                            label_results.append(entity)
                            matched = True
                            break
        
        # If only term_type is specified (no ID or label), add any matching entity
        if term_type and not entity_id and not label and not matched:
            results.append(entity)
    
    # Special case: if only term_type is specified, return all entities that matched the type
    if term_type and not entity_id and not label:
        return results
    
    # Combine results with label matches first (if prioritizing labels)
    if prioritize_label:
        # Remove duplicates while preserving order
        seen = set()
        for entity in label_results + id_results:
            entity_id = entity.get('@id')
            if entity_id not in seen:
                seen.add(entity_id)
                results.append(entity)
    else:
        # Traditional approach: ID matches first, then label matches
        results = id_results + [e for e in label_results if e not in id_results]
    
    return results


In [ ]:
# Create a test knowledge base with a rich variety of entity types and formats
kb = LinkedDataKnowledge()

# Set up a context with various prefixes
test_context = {
    "schema": "https://schema.org/",
    "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
    "owl": "http://www.w3.org/2002/07/owl#",
    "ex": "http://example.org/",
    "foaf": "http://xmlns.com/foaf/0.1/"
}

# Create test data with diverse entity types and representation formats
test_data = {
    "@context": test_context,
    "@graph": [
        # Standard class with simple type
        {
            "@id": "https://schema.org/Person",
            "@type": "rdfs:Class",
            "rdfs:label": "Person",
            "rdfs:comment": "A person (alive, dead, undead, or fictional)."
        },
        # Class with multiple types
        {
            "@id": "http://xmlns.com/foaf/0.1/Person",
            "@type": ["rdfs:Class", "owl:Class"],
            "rdfs:label": "Person",
            "owl:equivalentClass": {"@id": "https://schema.org/Person"}
        },
        # Entity with opaque URI
        {
            "@id": "ex:12345",
            "@type": "rdfs:Class",
            "rdfs:label": "Organization",
            "rdfs:comment": "An organization such as a school, NGO, corporation, club, etc."
        },
        # Entity with complex label structure
        {
            "@id": "http://example.org/Employee",
            "@type": "rdfs:Class",
            "rdfs:label": [
                {"@value": "Employee", "@language": "en"},
                {"@value": "Empleado", "@language": "es"}
            ],
            "rdfs:subClassOf": {"@id": "https://schema.org/Person"}
        },
        # Property with domain and range
        {
            "@id": "http://example.org/name",
            "@type": "rdf:Property",
            "rdfs:label": "name",
            "rdfs:domain": {"@id": "https://schema.org/Person"},
            "rdfs:range": {"@id": "xsd:string"}
        },
        # Entity with full URI as type
        {
            "@id": "ex:SpecialPerson",
            "@type": "http://www.w3.org/2000/01/rdf-schema#Class",
            "rdfs:label": "Special Person"
        },
        # Entity with mixed case label
        {
            "@id": "ex:CaseSensitive",
            "@type": "rdfs:Class",
            "rdfs:label": "CamelCaseLabel"
        }
    ]
}

kb.data = test_data

print("===== TESTING _has_type FUNCTION =====")

# Test 1: Exact match with prefixed name
entity = kb.data["@graph"][0]  # schema:Person with type rdfs:Class
result = kb._has_type(entity, "rdfs:Class")
print(f"Test 1 - Exact match with prefixed name: {result}")

# Test 2: Local name match
result = kb._has_type(entity, "Class")
print(f"Test 2 - Local name match: {result}")

# Test 3: Full URI match
result = kb._has_type(entity, "http://www.w3.org/2000/01/rdf-schema#Class")
print(f"Test 3 - Full URI match: {result}")

# Test 4: Multiple types
entity = kb.data["@graph"][1]  # foaf:Person with types rdfs:Class, owl:Class
result = kb._has_type(entity, "owl:Class")
print(f"Test 4 - Multiple types: {result}")

# Test 5: Full URI in entity type
entity = kb.data["@graph"][5]  # ex:SpecialPerson with type http://www.w3.org/2000/01/rdf-schema#Class
result = kb._has_type(entity, "rdfs:Class")
print(f"Test 5 - Full URI in entity type: {result}")

# Test 6: Non-matching type
entity = kb.data["@graph"][0]  # schema:Person with type rdfs:Class
result = kb._has_type(entity, "owl:Thing")
print(f"Test 6 - Non-matching type: {result}")

# Test 7: Empty or None inputs
result = kb._has_type({}, "rdfs:Class")
print(f"Test 7a - Empty entity: {result}")
result = kb._has_type(entity, "")
print(f"Test 7b - Empty type string: {result}")

print("\n===== TESTING find_entity FUNCTION =====")

# Test 8: Find by exact URI
result = kb.find_entity(entity_id="https://schema.org/Person")
print(f"Test 8 - Find by exact URI: Found {len(result)} entities")
if result:
    print(f"  - {result[0].get('@id')} ({result[0].get('rdfs:label')})")

# Test 9: Find by local name
result = kb.find_entity(entity_id="Person")
print(f"Test 9 - Find by local name: Found {len(result)} entities")
for entity in result:
    print(f"  - {entity.get('@id')} ({entity.get('rdfs:label')})")

# Test 10: Find by label
result = kb.find_entity(label="Person")
print(f"Test 10 - Find by label: Found {len(result)} entities")
for entity in result:
    print(f"  - {entity.get('@id')} ({entity.get('rdfs:label')})")

# Test 11: Find by type
result = kb.find_entity(term_type="rdfs:Class")
print(f"Test 11 - Find by type: Found {len(result)} entities")
for entity in result:
    print(f"  - {entity.get('@id')} ({entity.get('rdfs:label')})")

# Test 12: Find by language-tagged label
result = kb.find_entity(label="Empleado")
print(f"Test 12 - Find by language-tagged label: Found {len(result)} entities")
for entity in result:
    print(f"  - {entity.get('@id')}")

# Test 13: Case-insensitive search (default)
result = kb.find_entity(label="camelcaselabel")
print(f"Test 13 - Case-insensitive search: Found {len(result)} entities")
for entity in result:
    print(f"  - {entity.get('@id')} ({entity.get('rdfs:label')})")

# Test 14: Case-sensitive search
result = kb.find_entity(label="camelcaselabel", case_sensitive=True)
print(f"Test 14 - Case-sensitive search: Found {len(result)} entities")
for entity in result:
    print(f"  - {entity.get('@id')} ({entity.get('rdfs:label')}) - Should be empty")

# Test 15: Combined criteria (ID and type)
result = kb.find_entity(entity_id="Person", term_type="rdfs:Class")
print(f"Test 15 - Combined criteria: Found {len(result)} entities")
for entity in result:
    print(f"  - {entity.get('@id')} ({entity.get('rdfs:label')})")

# Test 16: Partial label match
result = kb.find_entity(label="Org")
print(f"Test 16 - Partial label match: Found {len(result)} entities")
for entity in result:
    print(f"  - {entity.get('@id')} ({entity.get('rdfs:label')})")

# Test 17: Find by opaque URI with label
result = kb.find_entity(label="Organization")
print(f"Test 17 - Find by opaque URI: Found {len(result)} entities")
for entity in result:
    print(f"  - {entity.get('@id')} ({entity.get('rdfs:label')})")

# Test 18: Find property by type
result = kb.find_entity(term_type="rdf:Property")
print(f"Test 18 - Find property by type: Found {len(result)} entities")
for entity in result:
    print(f"  - {entity.get('@id')} ({entity.get('rdfs:label')})")


===== TESTING _has_type FUNCTION =====
Test 1 - Exact match with prefixed name: True
Test 2 - Local name match: True
Test 3 - Full URI match: True
Test 4 - Multiple types: True
Test 5 - Full URI in entity type: True
Test 6 - Non-matching type: False
Test 7a - Empty entity: False
Test 7b - Empty type string: False

===== TESTING find_entity FUNCTION =====
Test 8 - Find by exact URI: Found 1 entities
  - https://schema.org/Person (Person)
Test 9 - Find by local name: Found 3 entities
  - https://schema.org/Person (Person)
  - http://xmlns.com/foaf/0.1/Person (Person)
  - ex:SpecialPerson (Special Person)
Test 10 - Find by label: Found 3 entities
  - https://schema.org/Person (Person)
  - http://xmlns.com/foaf/0.1/Person (Person)
  - ex:SpecialPerson (Special Person)
Test 11 - Find by type: Found 6 entities
  - https://schema.org/Person (Person)
  - http://xmlns.com/foaf/0.1/Person (Person)
  - ex:12345 (Organization)
  - http://example.org/Employee ([{'@value': 'Employee', '@language': '

In [ ]:
#| eval: false
# Then create a test function that doubles as documentation
def test_find_entity():
    "Test finding entities by ID, type, and label"
    # Create a test LinkedDataKnowledge object
    data = {
        '@graph': [
            {'@id': 'http://example.org/entity1', 'rdfs:label': 'Entity One'},
            {'@id': 'http://example.org/entity2', 'rdfs:label': 'Entity Two'},
            {'@id': 'http://example.org/entity3', 'rdfs:label': 'Entity Three', '@type': 'Class'},
        ]
    }
    
    kb = LinkedDataKnowledge(data)
    
    # Test finding by ID - this serves as documentation example too
    result1 = kb.find_entity(entity_id='entity1')
    test_eq(len(result1), 1)
    test_eq(result1[0]['@id'], 'http://example.org/entity1')
    
    # Test finding by partial ID
    result2 = kb.find_entity(entity_id='entity')
    test_eq(len(result2), 3)
    
    # Test finding by label
    result3 = kb.find_entity(label='Entity Two')
    test_eq(len(result3), 1)
    test_eq(result3[0]['@id'], 'http://example.org/entity2')
    
    # Test filtering by type
    result4 = kb.find_entity(entity_id='entity', term_type='Class')
    test_eq(len(result4), 1)
    test_eq(result4[0]['@id'], 'http://example.org/entity3')
    
    # Test with non-existent entity
    result5 = kb.find_entity(entity_id='nonexistent')
    test_eq(len(result5), 0)
    
    # You can also display a rich representation for documentation
    from IPython.display import display, Markdown
    display(Markdown("### Sample entity found:"))
    if result1:
        display(Markdown(f"**ID**: {result1[0]['@id']}  \n**Label**: {result1[0]['rdfs:label']}"))


In [ ]:
#| eval: false
test_find_entity()

### Sample entity found:

**ID**: http://example.org/entity1  
**Label**: Entity One

In [ ]:
#| export
@patch
def get_entity_description(self:LinkedDataKnowledge, entity:dict) -> str:
    "Get a formatted description of an entity"
    if not entity:
        return "No entity provided"
    
    lines = []
    
    # Entity ID
    entity_id = entity.get('@id', 'Unknown ID')
    lines.append(f"## Entity: {entity_id}")
    
    # Entity type
    entity_type = entity.get('@type', [])
    if not isinstance(entity_type, list):
        entity_type = [entity_type]
    lines.append(f"**Type**: {', '.join(entity_type)}")
    
    # Labels
    labels = []
    for key, value in entity.items():
        if 'label' in key.lower():
            if isinstance(value, list):
                for item in value:
                    if isinstance(item, dict) and '@value' in item:
                        labels.append(f"{item.get('@value')} ({item.get('@language', 'no language')})")
                    else:
                        labels.append(str(item))
            else:
                labels.append(str(value))
    
    if labels:
        lines.append(f"**Labels**: {', '.join(labels)}")
    
    # Comments/Definitions
    comments = []
    for key, value in entity.items():
        if 'comment' in key.lower() or 'definition' in key.lower():
            if isinstance(value, list):
                for item in value:
                    if isinstance(item, dict) and '@value' in item:
                        comments.append(item.get('@value'))
                    else:
                        comments.append(str(item))
            else:
                comments.append(str(value))
    
    if comments:
        lines.append("\n**Definition**:")
        for comment in comments:
            lines.append(f"- {comment}")
    
    # Relationships
    relationships = []
    for key, value in entity.items():
        if key not in ['@id', '@type'] and not any(x in key.lower() for x in ['label', 'comment', 'definition']):
            relationships.append((key, value))
    
    if relationships:
        lines.append("\n**Relationships**:")
        for rel_name, rel_value in relationships:
            if isinstance(rel_value, list):
                lines.append(f"- {rel_name}:")
                for item in rel_value:
                    if isinstance(item, dict) and '@id' in item:
                        lines.append(f"  - {item['@id']}")
                    elif isinstance(item, dict) and '@value' in item:
                        lines.append(f"  - {item['@value']}")
                    else:
                        lines.append(f"  - {item}")
            else:
                if isinstance(rel_value, dict) and '@id' in rel_value:
                    lines.append(f"- {rel_name}: {rel_value['@id']}")
                elif isinstance(rel_value, dict) and '@value' in rel_value:
                    lines.append(f"- {rel_name}: {rel_value['@value']}")
                else:
                    lines.append(f"- {rel_name}: {rel_value}")
    
    return "\n".join(lines)


In [ ]:
#| export
@patch
def query(self:LinkedDataKnowledge,
         query_type:str, # Type of query: "property", "type", "value"
         query_value:str # Value to search for
         ) -> list:
    """Queries JSON-LD data for specific properties, types, or values."""
    # First, try to expand the query value if it's a prefixed term
    expanded_value = query_value
    
    # Handle prefixed terms (like schema:Person)
    if ':' in query_value and not query_value.startswith(('http://', 'https://')):
        prefix, local = query_value.split(':', 1)
        if prefix in self.data.get('@context', {}):
            prefix_uri = self.data['@context'][prefix]
            if isinstance(prefix_uri, str):
                if prefix_uri.endswith('/') or prefix_uri.endswith('#'):
                    expanded_value = f"{prefix_uri}{local}"
                else:
                    expanded_value = f"{prefix_uri}/{local}"
    
    # Handle non-prefixed terms that might be defined in context
    elif not query_value.startswith(('http://', 'https://')):
        # Check if term is defined in @context
        if query_value in self.data.get('@context', {}):
            term_def = self.data['@context'][query_value]
            if isinstance(term_def, dict) and '@id' in term_def:
                expanded_value = term_def['@id']
            elif isinstance(term_def, str):
                expanded_value = term_def
        
        # Check if there's a @vocab that would apply
        elif '@vocab' in self.data.get('@context', {}):
            vocab = self.data['@context']['@vocab']
            if vocab.endswith('/') or vocab.endswith('#'):
                expanded_value = f"{vocab}{query_value}"
            else:
                expanded_value = f"{vocab}/{query_value}"
    
    # Convert JSON-LD to expanded form for consistent access
    expanded = jsonld.expand(self.data)
    results = []
    
    if query_type == "property":
        # Find entities with a specific property (using both original and expanded)
        for entity in expanded:
            if query_value in entity or expanded_value in entity:
                results.append(entity)
    elif query_type == "type":
        # Find entities of a specific type (using both original and expanded)
        for entity in expanded:
            if '@type' not in entity:
                continue
                
            types = entity['@type']
            if not isinstance(types, list):
                types = [types]
                
            if query_value in types or expanded_value in types:
                results.append(entity)
    elif query_type == "value":
        # Find entities with properties containing a specific value
        for entity in expanded:
            for prop, values in entity.items():
                if prop == "@id" or prop == "@type":
                    continue
                for val in values:
                    if "@value" in val and val["@value"] == query_value:
                        results.append(entity)
                        break
    
    return results


In [ ]:
 #| export
def describe(self, path:str="") -> str:
    "Describe the structure at the given path in a human-readable format"
    # Implementation that combines exploration and visualization
    pass

In [ ]:
#| export
def view(self, entity_id:str=None, term_type:str=None, label:str=None) -> None:
    "Find and display entities in a rich format"
    entities = self.find_entity(entity_id, term_type, label)
    if not entities:
        print(f"No entities found matching the criteria")
        return
    
    for entity in entities:
        display(Markdown(self.get_entity_description(entity)))

In [ ]:
#| test
def test_initialization():
    kb = LinkedDataKnowledge()
    test_eq(kb.data, {"@context": {}, "@graph": []})
    
    test_data = {"@context": {"schema": "https://schema.org/"}, "@graph": [{"@id": "test"}]}
    kb2 = LinkedDataKnowledge(test_data)
    test_eq(kb2.data, test_data)

In [ ]:
test_initialization()

In [ ]:
#| test
def test_find_entity():
    test_kb = LinkedDataKnowledge({
        "@context": {},
        "@graph": [
            {
                "@id": "http://example.org/Person",
                "@type": "rdfs:Class",
                "rdfs:label": "Person"
            },
            {
                "@id": "http://example.org/name",
                "@type": "rdf:Property",
                "rdfs:label": "name"
            }
        ]
    })
    # Test find by ID
    person_results = test_kb.find_entity(entity_id="Person")
    test_eq(len(person_results), 1)
    test_eq(person_results[0]['@id'], "http://example.org/Person")

    # Test find by type
    class_results = test_kb.find_entity(term_type="rdfs:Class")
    test_eq(len(class_results), 1)
    test_eq(class_results[0]['@id'], "http://example.org/Person")

    # Test find by label
    label_results = test_kb.find_entity(label="Person")
    test_eq(len(label_results), 1)
    test_eq(label_results[0]['@id'], "http://example.org/Person")

In [ ]:
def test_find_entity():
    # Create a test LinkedDataKnowledge object
    data = {
        '@graph': [
            {'@id': 'http://example.org/entity1', 'rdfs:label': 'Entity One'},
            {'@id': 'http://example.org/entity2', 'rdfs:label': 'Entity Two'},
            {'@id': 'http://example.org/entity3', 'rdfs:label': 'Entity Three', '@type': 'Class'},
        ]
    }
    
    knowledge = LinkedDataKnowledge(data)
    
    # Test finding by ID
    result1 = knowledge.find_entity(entity_id='entity1')
    assert len(result1) == 1
    assert result1[0]['@id'] == 'http://example.org/entity1'
    
    # Test finding by partial ID
    result2 = knowledge.find_entity(entity_id='entity')
    assert len(result2) == 3
    
    # Test finding by label
    result3 = knowledge.find_entity(label='Entity Two')
    assert len(result3) == 1
    assert result3[0]['@id'] == 'http://example.org/entity2'
    
    # Test filtering by type
    result4 = knowledge.find_entity(entity_id='entity', term_type='Class')
    assert len(result4) == 1
    assert result4[0]['@id'] == 'http://example.org/entity3'
    
    # Test with non-existent entity
    result5 = knowledge.find_entity(entity_id='nonexistent')
    assert len(result5) == 0

In [ ]:
#| test
def test_get_entity_description():
    data = {
        '@graph': [
            {
                '@id': 'http://example.org/entity1', 
                'rdfs:label': 'Test Entity',
                'rdfs:comment': 'This is a test entity',
                '@type': 'Class'
            },
            {
                '@id': 'http://example.org/entity2', 
                'rdfs:label': [{'@value': 'Multi-language Entity', '@language': 'en'}],
                'skos:definition': 'Entity with complex properties'
            }
        ]
    }
    
    kb = LinkedDataKnowledge(data)
    
    # Test basic description
    entity1 = kb.find_entity(entity_id='entity1')[0]
    desc1 = kb.get_entity_description(entity1)
    assert 'Test Entity' in desc1
    assert 'This is a test entity' in desc1
    assert 'Class' in desc1
    
    # Test with complex properties
    entity2 = kb.find_entity(entity_id='entity2')[0]
    desc2 = kb.get_entity_description(entity2)
    assert 'Multi-language Entity' in desc2
    assert 'Entity with complex properties' in desc2

In [ ]:
test_get_entity_description()

In [ ]:
test_find_entity()

In [ ]:
#| eval: False
def test_markdown_representation():
    kb = LinkedDataKnowledge({
        "@context": {"schema": "https://schema.org/"},
        "@graph": [
            {
                "@id": "http://example.org/person1",
                "@type": "schema:Person",
                "schema:name": "Alice Smith"
            }
        ]
    })
    
    md = kb._repr_markdown_()
    assert('LinkedDataKnowledge' in md)
    assert('Context' in md)
    assert('Graph' in md)
    assert('schema:Person' in md)
    
test_markdown_representation()

In [ ]:
#| eval: false
from IPython.display import Markdown, display

def test_markdown_display():
    """Test the markdown display of entity descriptions"""
    data = {
        '@graph': [
            {
                '@id': 'http://example.org/entity1', 
                'rdfs:label': 'Test Entity',
                'rdfs:comment': 'This is a test entity',
                '@type': 'Class'
            }
        ]
    }
    
    kb = LinkedDataKnowledge(data)
    entity = kb.find_entity(entity_id='entity1')[0]
    desc = kb.get_entity_description(entity)
    
    # Display the markdown representation
    display(Markdown(desc))
    
    return "Markdown display test passed"

# Run the test
test_markdown_display()

## Entity: http://example.org/entity1
**Type**: Class
**Labels**: Test Entity

**Definition**:
- This is a test entity

'Markdown display test passed'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()